# Kannada-MNIST

## Submission

Train a fresh CNN on **all 60,000 TRAIN and all 10,240 Dig-MNIST images** for ten fixed epochs, then predict the unlabeled competition test set.

- [Competition](https://www.kaggle.com/competitions/Kannada-MNIST)
- [Data](https://www.kaggle.com/competitions/Kannada-MNIST/data)
- [Experimental comparison](report.ipynb)

Use the same CNN, Adam learning rate 0.001, batch size 256, and seed 42 as the experiments. Initialize new weights; do not continue training an experimental checkpoint. Each image appears once per epoch with ordinary shuffled batches and no source reweighting.

**There is no validation holdout in this final fit.** Former experimental validation images are now training data. Training metrics do not estimate generalization, and `test.csv` has no labels. Only a completed Kaggle competition submission can provide the competition score. Full-data training is intended to use all available labeled examples; an improvement is not guaranteed.

Run with the Kaggle GPU environment and `/usr/bin/python3` locally. On Kaggle, attach the competition data and disable internet. This notebook writes predictions but never uploads or submits them automatically.

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
import hashlib
import json
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import torch
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, TensorDataset

# fix seed to reproduce results

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# resource check
print("Host:", os.uname().nodename)
print("Python:", sys.version)
print("Interpreter:", sys.executable)

assert torch.cuda.is_available()
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

print(
    {
        p: version(p)
        for p in ["torch", "numpy", "pandas", "scikit-learn", "plotly", "kagglehub"]
    }
)

probe = torch.arange(16, dtype=torch.float32, device=device)
assert (probe @ probe).item() == 1240.0
print("CUDA computation passed")

# Embed Plotly assets so notebook outputs can be viewed without a CDN.
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Data preparation

Use KaggleHub for missing CSVs. Reuse local files, keep original row order, and preserve sample-submission IDs.

In [ ]:
import kagglehub
from dotenv import load_dotenv

COMPETITION = "Kannada-MNIST"
repo_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "competitions" / COMPETITION).is_dir()
    ),
    None,
)
project_dir = (
    repo_dir / "competitions" / COMPETITION
    if repo_dir is not None
    else Path.cwd() if Path.cwd().name == COMPETITION else Path.cwd() / COMPETITION
)
output_dir = project_dir / "output"

if repo_dir is not None:
    load_dotenv(repo_dir / ".env", override=False)

required_files = ("train.csv", "test.csv", "sample_submission.csv", "Dig-MNIST.csv")
data_dir = project_dir / "data"
if not all((data_dir / name).is_file() for name in required_files):
    data_dir = Path(
        kagglehub.competition_download(
            COMPETITION,
            output_dir=str(data_dir.resolve()),
        )
    )

missing = [name for name in required_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing competition files in {data_dir}: {missing}")

print("Competition data:", data_dir.resolve())
print("Output directory:", output_dir.resolve())

output_dir = project_dir / "output" / "submission"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
train = pd.read_csv(data_dir / "train.csv", dtype=np.int16)
dig_mnist = pd.read_csv(data_dir / "Dig-MNIST.csv", dtype=np.int16)
submission_template = pd.read_csv(data_dir / "sample_submission.csv")
test = pd.read_csv(data_dir / "test.csv")

pixels = [f"pixel{i}" for i in range(784)]
assert train.columns.tolist() == ["label", *pixels]
assert dig_mnist.columns.tolist() == ["label", *pixels]
assert submission_template.columns.tolist() == ["id", "label"]
assert test.columns.tolist() == ["id", *pixels]
assert len(test) == len(submission_template)
assert test["id"].is_unique
assert test["id"].equals(submission_template["id"])

for name, frame in [("train", train), ("Dig-MNIST", dig_mnist), ("test", test)]:
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert frame[pixels].min().min() >= 0 and frame[pixels].max().max() <= 255
    if "label" in frame:
        assert frame["label"].between(0, 9).all()

display(
    pd.DataFrame(
        [
            {"file": "train.csv", "rows": len(train), "columns": len(train.columns)},
            {
                "file": "Dig-MNIST.csv",
                "rows": len(dig_mnist),
                "columns": len(dig_mnist.columns),
            },
            {"file": "test.csv", "rows": len(test), "columns": len(test.columns)},
            {
                "file": "sample_submission.csv",
                "rows": len(submission_template),
                "columns": len(submission_template.columns),
            },
        ]
    )
)
display(train.head())
display(submission_template.head())


## Full-data CNN training

Concatenate both complete labeled datasets with PyTorch `ConcatDataset`. Do not create split indices or evaluate either training source as a held-out set. Fit a newly initialized model for exactly ten epochs with no early stopping or checkpoint selection.

In [ ]:
def image_dataset(frame):
    return TensorDataset(
        torch.from_numpy(frame[pixels].to_numpy(dtype=np.float32)).reshape(
            -1, 1, 28, 28
        )
        / 255.0,
        torch.from_numpy(frame["label"].to_numpy(dtype=np.int64)),
    )


train_dataset = ConcatDataset([image_dataset(train), image_dataset(dig_mnist)])
assert len(train_dataset) == len(train) + len(dig_mnist)
print("Full labeled training rows:", len(train_dataset))


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=256,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )


train_loader = make_loader(train_dataset, shuffle=True)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


EPOCHS = 10
history = []

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    total_loss = 0.0
    total_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

    row = {
        "epoch": epoch,
        "training_loss": total_loss / len(train_dataset),
        "training_accuracy": total_correct / len(train_dataset),
        "seconds": time.perf_counter() - started,
    }
    history.append(row)
    print(row)

history_df = pd.DataFrame(history)
assert np.isfinite(history_df.select_dtypes(include="number")).all().all()
display(history_df)


## Training curves

Loss and accuracy below are measured on training batches while weights change. They describe optimization only; they are not validation or competition scores.

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Training loss", "Training accuracy"]
)
fig.add_trace(
    go.Scatter(
        x=history_df["epoch"], y=history_df["training_loss"], name="Training loss"
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=history_df["epoch"],
        y=history_df["training_accuracy"],
        name="Training accuracy",
    ),
    row=1,
    col=2,
)
fig.update_xaxes(title_text="Epoch", dtick=1)
fig.update_yaxes(title_text="Cross-entropy", row=1, col=1)
fig.update_yaxes(title_text="Accuracy", tickformat=".1%", row=1, col=2)
fig.update_layout(title="Full-data fit: training metrics only", height=420)
fig.show()


## Test inference and submission artifacts

Preserve the original test IDs and `id,label` column order. Save the model, input hashes, settings, and training history under `output/submission/`. On Kaggle, write `submission.csv` to `/kaggle/working/` for the competition submitter. No test accuracy, held-out error gallery, or held-out F1 is reported for this full-data fit.

In [ ]:
def predict_probabilities(dataset):
    probabilities = []
    model.eval()
    with torch.inference_mode():
        for batch in make_loader(dataset):
            images = batch[0].to(device)
            probabilities.append(model(images).softmax(dim=1).cpu().numpy())
    return np.concatenate(probabilities)


test_dataset = TensorDataset(
    torch.from_numpy(test[pixels].to_numpy(dtype=np.float32)).reshape(-1, 1, 28, 28)
    / 255.0,
)
test_probabilities = predict_probabilities(test_dataset)
submission = submission_template.copy()
submission["label"] = test_probabilities.argmax(axis=1)
assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].equals(test["id"])
assert submission["label"].between(0, 9).all()
# Kaggle's competition submitter should receive a root-level output filename.
submission_path = (
    Path("/kaggle/working/submission.csv")
    if repo_dir is None and Path("/kaggle/working").is_dir()
    else output_dir / "submission.csv"
)
submission.to_csv(submission_path, index=False)
print("Competition submission file:", submission_path.resolve())
history_df.to_csv(output_dir / "training_history.csv", index=False)

training_manifest = {
    "method": "Full TRAIN + full Dig-MNIST; no validation holdout",
    "training_rows": {"TRAIN": len(train), "Dig-MNIST": len(dig_mnist)},
    "epochs": EPOCHS,
    "seed": SEED,
    "batch_size": 256,
    "learning_rate": 0.001,
    "train_csv_sha256": hashlib.sha256(
        (data_dir / "train.csv").read_bytes()
    ).hexdigest(),
    "dig_csv_sha256": hashlib.sha256(
        (data_dir / "Dig-MNIST.csv").read_bytes()
    ).hexdigest(),
    "versions": {
        name: version(name) for name in ["torch", "numpy", "pandas", "kagglehub"]
    },
}
(output_dir / "training_manifest.json").write_text(
    json.dumps(training_manifest, indent=2) + "\n"
)
torch.save(
    {"state_dict": model.state_dict(), **training_manifest}, output_dir / "model.pt"
)
print("Saved full-data artifacts:", output_dir.resolve())
display(submission.head())
